## Notebook Purpose

This notebook (**Notebook 1 of 4** in the research pipeline) generates the synthetic
fine-tuning dataset used later to train and evaluate Small Language Models (SLMs) on
Project Management (PM) tasks: **Task Design, Task Allocation, Resource Planning,
Sprint Planning**.

**Methodology:** A hybrid generation strategy is used — a portion of examples are
seeded from real Jira-style issues (`REAL_JIRA_SEEDS`) to ground the dataset in
realistic scenarios, and the remainder are fully synthetic, both generated by calling
the Inception Labs `mercury-2` model through their API.

**Input:** No external file input — seed examples are hard-coded in this notebook.

**Output:** `dataset/pm_finetune_dataset.jsonl` (and train/val/test splits), formatted
as Qwen2-style chat completions, ready to be consumed by the fine-tuning notebook
(`with-results-qwen-finetune-eval-compare-v1.ipynb`) and by the benchmarking notebook
after further processing in `journal2_helper_functions.ipynb`.

**Why this step is necessary:** Both the SLM benchmark and the QLoRA fine-tuning
experiment require a labeled, schema-consistent PM dataset. No off-the-shelf dataset
covers all four PM task types in a structured JSON format, so this notebook builds one.


In [1]:
!pip install requests datasets tqdm pandas --quiet

### Inception API — PM Synthetic Dataset Generator

### Install Dependencies

Installs the Python packages needed to call the Inception API and process the
resulting data: `requests` (HTTP calls), `datasets` (HF dataset utilities),
`tqdm` (progress bars), and `pandas` (tabular inspection).


In [ ]:
"""
==============================================================================
  PM Dataset Preparation for Fine-tuning Qwen2.5-7B
  -------------------------------------------------------
  Thesis: Project Management SLM Fine-tuning
  Tasks:  Task Design | Task Allocation | Resource Planning | Sprint Planning
  Method: Hybrid (Real Jira-seeded + Synthetic via Inception API)
  Target: ~2000 examples (500 per task type)
  API:    Inception Labs — mercury-2 model (100 req/min limit)
  Output: dataset/pm_finetune_dataset.jsonl  (Qwen2 chat format)

  HOW TO RUN (Google Colab):
      1. Set your INCEPTION_API_KEY below
      2. Run all cells sequentially
      3. Dataset saved to /content/dataset/
==============================================================================
"""

import os
import json
import time
import random
import hashlib
import requests
import threading
import pandas as pd
from tqdm import tqdm
from collections import deque
from datasets import Dataset
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

INCEPTION_API_KEY  = "your-key"   # <-- Replace this
INCEPTION_BASE_URL = "https://api.inceptionlabs.ai/v1/chat/completions"
GENERATOR_MODEL    = "mercury-2"

OUTPUT_DIR         = Path("dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

# Examples per task type
EXAMPLES_PER_TASK  = 500   # x4 tasks = 2000 total
REAL_RATIO         = 0.30  # 30% real Jira-seeded, 70% fully synthetic

# Inception API rate limit: 100 requests/minute
# We use a sliding window to track and throttle requests safely
MAX_REQUESTS_PER_MIN = 95   # stay slightly under the 100 hard limit
RATE_WINDOW_SEC      = 60.0

print(f"Config loaded.")
print(f"Target: {EXAMPLES_PER_TASK * 4} examples across 4 task types.")
print(f"Rate limit: {MAX_REQUESTS_PER_MIN} req/min (Inception allows 100).")

Config loaded.
Target: 2000 examples across 4 task types.
Rate limit: 95 req/min (Inception allows 100).


### Pipeline Configuration & Imports

Sets the run parameters for the whole dataset-generation pipeline: how many examples
to generate per task type, the real-vs-synthetic mix ratio, the target model, and the
output directory. Also configures the Inception API key. Keeping all constants in one
place makes it easy to do quick smoke tests (few examples) before committing to a full
~2000-example run.


In [13]:
# ============================================================
# RATE LIMITER  (sliding window — thread-safe)
# ============================================================

class SlidingWindowRateLimiter:
    """
    Keeps timestamps of the last N requests in a deque.
    Before each request it blocks until the oldest timestamp
    falls outside the 60-second window, ensuring we never
    exceed MAX_REQUESTS_PER_MIN.
    """
    def __init__(self, max_requests: int, window_seconds: float):
        self.max_requests = max_requests
        self.window       = window_seconds
        self._timestamps  = deque()
        self._lock        = threading.Lock()

    def acquire(self):
        """Block until a request slot is available."""
        with self._lock:
            now = time.monotonic()
            # Evict timestamps older than the window
            while self._timestamps and now - self._timestamps[0] >= self.window:
                self._timestamps.popleft()

            if len(self._timestamps) >= self.max_requests:
                # Must wait until the oldest request is outside the window
                sleep_for = self.window - (now - self._timestamps[0]) + 0.05
                if sleep_for > 0:
                    time.sleep(sleep_for)
                # Re-evict after sleeping
                now = time.monotonic()
                while self._timestamps and now - self._timestamps[0] >= self.window:
                    self._timestamps.popleft()

            self._timestamps.append(time.monotonic())

rate_limiter = SlidingWindowRateLimiter(MAX_REQUESTS_PER_MIN, RATE_WINDOW_SEC)
print("Rate limiter initialised.")

Rate limiter initialised.


### Rate Limiter

A thread-safe sliding-window rate limiter used to keep API calls under Inception's
100 requests/minute cap. Because example generation is I/O-bound (waiting on API
responses), later cells parallelize requests across threads — this class ensures that
parallelism doesn't exceed the provider's rate limit and trigger `429` errors.

**How it works:** every time `acquire()` is called it records a timestamp; if the
number of timestamps within the trailing 60-second window would exceed
`max_requests`, it blocks until enough time has passed for the oldest timestamp to
"expire" out of the window.


In [14]:
REAL_JIRA_SEEDS = [
    # ── AgriTech ────────────────────────────────────────────
    {"project": "Smart Farm Platform",  "role_context": "Data Engineer",
     "issue": "Build soil moisture time-series ingestion pipeline from 10k distributed sensors",
     "domain": "agritech",            "complexity": "high"},
    {"project": "Smart Farm Platform",  "role_context": "ML Engineer",
     "issue": "Implement crop yield prediction model using satellite NDVI imagery and weather data",
     "domain": "agritech",            "complexity": "high"},
    {"project": "AgriMarket Platform",  "role_context": "Backend Engineer",
     "issue": "Build commodity price alert engine with configurable threshold rules per farmer profile",
     "domain": "agritech",            "complexity": "medium"},
 
    # ── Climate / Sustainability ─────────────────────────────
    {"project": "Carbon Accounting SaaS","role_context": "Backend Engineer",
     "issue": "Implement Scope 3 emission calculation engine following GHG Protocol methodology",
     "domain": "sustainability",      "complexity": "high"},
    {"project": "Carbon Accounting SaaS","role_context": "Data Engineer",
     "issue": "Build supplier emission data collection pipeline with validation and audit trail",
     "domain": "sustainability",      "complexity": "medium"},
    {"project": "Renewable Energy OS",  "role_context": "DevOps Engineer",
     "issue": "Set up multi-region data pipeline for aggregating solar panel telemetry at 1s intervals",
     "domain": "sustainability",      "complexity": "high"},
 
    # ── Blockchain / Web3 ───────────────────────────────────
    {"project": "DeFi Protocol",        "role_context": "Backend Engineer",
     "issue": "Implement on-chain liquidation bot with MEV-aware gas price strategy",
     "domain": "blockchain",          "complexity": "high"},
    {"project": "NFT Marketplace",      "role_context": "Backend Engineer",
     "issue": "Build lazy minting service with gasless meta-transaction support via EIP-2771",
     "domain": "blockchain",          "complexity": "high"},
    {"project": "Web3 Wallet",          "role_context": "Security Engineer",
     "issue": "Implement multi-party computation key signing to eliminate single private key exposure",
     "domain": "blockchain",          "complexity": "high"},
 
    # ── Autonomous Vehicles / Robotics ───────────────────────
    {"project": "AV Simulation Platform","role_context": "ML Engineer",
     "issue": "Build scenario generation pipeline for edge-case adversarial driving situations",
     "domain": "autonomous-systems",  "complexity": "high"},
    {"project": "Fleet Management OS",  "role_context": "Backend Engineer",
     "issue": "Implement real-time route optimisation for 500-vehicle fleet with dynamic rerouting",
     "domain": "autonomous-systems",  "complexity": "high"},
    {"project": "AV Simulation Platform","role_context": "Data Engineer",
     "issue": "Build petabyte-scale sensor log storage with time-indexed retrieval under 200ms",
     "domain": "autonomous-systems",  "complexity": "high"},
 
    # ── Social Platform ─────────────────────────────────────
    {"project": "Creator Platform",     "role_context": "Backend Engineer",
     "issue": "Implement fan subscription billing with proration, pausing, and dunning management",
     "domain": "social",              "complexity": "medium"},
    {"project": "Creator Platform",     "role_context": "ML Engineer",
     "issue": "Build real-time content moderation classifier for images and short-form video",
     "domain": "social",              "complexity": "high"},
    {"project": "Social Feed Service",  "role_context": "Backend Engineer",
     "issue": "Re-architect activity feed from pull to push fanout model to reduce read latency",
     "domain": "social",              "complexity": "high"},
 
    # ── Travel Tech ─────────────────────────────────────────
    {"project": "Flight Search Engine", "role_context": "Backend Engineer",
     "issue": "Implement fare caching layer with TTL-based invalidation for 50M daily searches",
     "domain": "travel",              "complexity": "high"},
    {"project": "Hotel Booking Platform","role_context": "Data Engineer",
     "issue": "Build dynamic pricing pipeline ingesting competitor rates and demand signals",
     "domain": "travel",              "complexity": "medium"},
 
    # ── Digital Health / Wearables ───────────────────────────
    {"project": "Wearable Health App",  "role_context": "ML Engineer",
     "issue": "Implement on-device arrhythmia detection model optimised for Apple Watch Neural Engine",
     "domain": "digital-health",      "complexity": "high"},
    {"project": "Wearable Health App",  "role_context": "Backend Engineer",
     "issue": "Build FHIR R4 compliant data export so patients can share wearable data with clinicians",
     "domain": "digital-health",      "complexity": "medium"},
 
    # ── Government / Civic Tech ─────────────────────────────
    {"project": "Open Data Portal",     "role_context": "Backend Engineer",
     "issue": "Implement WCAG 2.1 AA accessibility audit pipeline as part of CI/CD for all public pages",
     "domain": "civic-tech",          "complexity": "medium"},
    {"project": "Benefits Eligibility", "role_context": "Backend Engineer",
     "issue": "Build rules engine for dynamic eligibility determination across 200+ benefit programmes",
     "domain": "civic-tech",          "complexity": "high"},
]
 
TEAM_ROSTER_POOL = [
    {"name": "Arjun Sharma",    "role": "Senior Backend Engineer",  "skills": ["Python", "Django", "PostgreSQL", "Redis"], "capacity": 8},
    {"name": "Priya Nair",      "role": "Frontend Engineer",        "skills": ["React", "TypeScript", "CSS", "GraphQL"],  "capacity": 8},
    {"name": "Mohammed Al-Razi","role": "DevOps Engineer",          "skills": ["Kubernetes", "Terraform", "CI/CD", "AWS"],"capacity": 8},
    {"name": "Sarah Chen",      "role": "ML Engineer",              "skills": ["PyTorch", "MLflow", "Python", "Spark"],   "capacity": 8},
    {"name": "Raj Patel",       "role": "QA Engineer",              "skills": ["Selenium", "Jest", "Cypress", "Postman"], "capacity": 6},
    {"name": "Lisa Nguyen",     "role": "UI/UX Designer",           "skills": ["Figma", "Prototyping", "User Research"],  "capacity": 8},
    {"name": "Omar Hassan",     "role": "Data Engineer",            "skills": ["Spark", "Airflow", "dbt", "Snowflake"],   "capacity": 8},
    {"name": "Ananya Reddy",    "role": "Security Engineer",        "skills": ["OWASP", "Penetration Testing", "OAuth2"], "capacity": 6},
    {"name": "Carlos Mendes",   "role": "Mobile Engineer",          "skills": ["React Native", "Swift", "Kotlin"],        "capacity": 8},
    {"name": "Fatima Malik",    "role": "Junior Backend Engineer",  "skills": ["Python", "Flask", "MySQL"],               "capacity": 8},
]
 
print(f"Loaded {len(REAL_JIRA_SEEDS)} Jira seeds | {len(TEAM_ROSTER_POOL)} team members.")

Loaded 21 Jira seeds | 10 team members.


### Real Jira-Style Seed Examples

Hand-curated seed issues spanning multiple domains (AgriTech, etc.), each with a
project name, role context, issue description, domain, and complexity rating. These
seeds anchor a portion of the generated dataset in realistic, human-written scenarios
rather than being 100% model-generated, which improves dataset diversity and realism.


In [15]:
# ============================================================
# SYSTEM PROMPT + PROMPT TEMPLATES
# ============================================================

SYSTEM_PROMPT = """You are an expert project manager and agile coach.
You produce precise, structured, and actionable project management artifacts.
Always output valid JSON. Do not include markdown code fences or extra text."""


def make_task_design_prompt(seed: dict, is_real: bool) -> str:
    if is_real:
        return f"""You are a project manager for the {seed['project']} project.

A developer has raised this issue: "{seed['issue']}"
Domain: {seed['domain']} | Complexity: {seed['complexity']}

Break this down into a well-structured task with full project management detail.

Return JSON with this exact structure:
{{
  "task_id": "TASK-XXX",
  "title": "short action-verb title",
  "description": "2-3 sentence detailed description",
  "acceptance_criteria": ["criterion 1", "criterion 2", "criterion 3"],
  "story_points": <fibonacci: 1,2,3,5,8,13>,
  "priority": "Critical|High|Medium|Low",
  "labels": ["label1", "label2"],
  "dependencies": ["describe any blocking tasks or leave empty array"],
  "definition_of_done": "clear measurable done state"
}}"""
    else:
        domains  = ["fintech", "healthcare SaaS", "e-commerce", "logistics",
                    "edtech", "IoT platform", "HR software", "CRM system"]
        features = ["authentication module", "payment gateway", "real-time notifications",
                    "reporting dashboard", "search and filtering", "API rate limiting",
                    "data export feature", "user onboarding flow", "audit logging",
                    "multi-tenant support", "webhook integration", "mobile push notifications"]
        return f"""You are a project manager for a {random.choice(domains)} startup.

Your team needs to build: {random.choice(features)}

Design a complete, actionable task for this feature.

Return JSON with this exact structure:
{{
  "task_id": "TASK-XXX",
  "title": "short action-verb title",
  "description": "2-3 sentence detailed description",
  "acceptance_criteria": ["criterion 1", "criterion 2", "criterion 3"],
  "story_points": <fibonacci: 1,2,3,5,8,13>,
  "priority": "Critical|High|Medium|Low",
  "labels": ["label1", "label2"],
  "dependencies": ["describe any blocking tasks or leave empty array"],
  "definition_of_done": "clear measurable done state"
}}"""


def make_task_allocation_prompt(seed: dict, team: list, is_real: bool) -> str:
    team_str  = json.dumps(team, indent=2)
    task_desc = seed['issue'] if is_real else f"Implement {random.choice(['OAuth2 login', 'CSV data export', 'email notification service', 'search indexing', 'caching layer', 'file upload service'])}"
    return f"""You are a project manager assigning tasks to team members.

TASK: {task_desc}
REQUIRED ROLE: {seed.get('role_context', 'Engineer')}
COMPLEXITY: {seed.get('complexity', 'medium')}

AVAILABLE TEAM:
{team_str}

Assign this task to the most suitable team member and explain your reasoning.

Return JSON with this exact structure:
{{
  "assigned_to": "Full Name",
  "role": "their role",
  "rationale": "2-3 sentences explaining WHY this person is best suited",
  "skill_match": ["matched skill 1", "matched skill 2"],
  "estimated_hours": <number>,
  "pairing_suggestion": "optional: another team member who could help or null",
  "risks": "any allocation risks or capacity concerns"
}}"""


def make_resource_planning_prompt(is_real: bool) -> str:
    sprint_count = random.randint(2, 4)
    team_size    = random.randint(3, 6)
    team_slice   = random.sample(TEAM_ROSTER_POOL, team_size)
    team_str     = json.dumps(
        [{"name": m["name"], "role": m["role"], "capacity_days": m["capacity"]} for m in team_slice],
        indent=2
    )
    project = random.choice(["Customer Portal Redesign", "API Gateway Migration",
                             "Mobile App v2.0", "Data Analytics Pipeline",
                             "Payment Service Refactor", "DevOps Automation Platform"])
    return f"""You are a project manager doing resource planning for the "{project}" project.

SPRINT DURATION: 2 weeks
NUMBER OF SPRINTS: {sprint_count}
TEAM:
{team_str}

Create a resource allocation plan across {sprint_count} sprints.

Return JSON with this exact structure:
{{
  "project": "{project}",
  "total_sprints": {sprint_count},
  "sprints": [
    {{
      "sprint_number": 1,
      "sprint_goal": "short goal statement",
      "allocations": [
        {{
          "member": "Full Name",
          "focus_area": "what they are working on",
          "allocated_days": <number>,
          "tasks": ["task 1", "task 2"]
        }}
      ],
      "total_capacity_days": <number>,
      "planned_velocity": <story points>
    }}
  ],
  "risks": ["risk 1", "risk 2"],
  "buffer_days": <number per sprint for meetings, reviews, bugs>
}}"""


def make_sprint_planning_prompt(is_real: bool) -> str:
    velocity  = random.randint(20, 45)
    team_size = random.randint(3, 5)
    backlog   = [
        {"id": f"TASK-{100+i}", "title": t, "story_points": sp, "priority": p}
        for i, (t, sp, p) in enumerate([
            ("User authentication with OAuth2",            5,  "Critical"),
            ("Product search with filters",                8,  "High"),
            ("Email notification service",                 3,  "High"),
            ("Admin dashboard - user management",          8,  "Medium"),
            ("Export data to CSV/Excel",                   3,  "Medium"),
            ("API rate limiting implementation",           5,  "High"),
            ("Fix checkout page loading performance",      3,  "Critical"),
            ("Mobile responsive design fixes",             5,  "Medium"),
            ("Integrate payment gateway",                  13, "Critical"),
            ("Write unit tests for auth module",           3,  "High"),
            ("Add pagination to product listing",          2,  "Medium"),
            ("Set up staging environment CI/CD",           5,  "High"),
            ("Database query optimization",                5,  "Medium"),
            ("Multi-language support (i18n)",              8,  "Low"),
            ("User profile page",                          3,  "Low"),
        ])
    ]
    random.shuffle(backlog)
    return f"""You are a Scrum Master facilitating sprint planning.

TEAM SIZE: {team_size} developers
SPRINT VELOCITY: {velocity} story points
SPRINT DURATION: 2 weeks

PRODUCT BACKLOG (prioritized):
{json.dumps(backlog, indent=2)}

Select the appropriate stories for this sprint, define the sprint goal, and plan capacity.

Return JSON with this exact structure:
{{
  "sprint_goal": "one-sentence goal that describes the business value",
  "selected_stories": [
    {{
      "id": "TASK-XXX",
      "title": "task title",
      "story_points": <number>,
      "assignee": "Developer N (role)",
      "notes": "any implementation notes"
    }}
  ],
  "total_story_points": <must be <= {velocity}>,
  "capacity_breakdown": {{
    "total_team_days": <number>,
    "planned_dev_days": <number>,
    "buffer_days": <number for meetings, reviews, unexpected>
  }},
  "deferred_stories": ["TASK-IDs deferred to next sprint"],
  "sprint_risks": ["risk 1", "risk 2"],
  "definition_of_ready_check": ["all stories have acceptance criteria", "dependencies resolved"]
}}"""


print("Prompt templates loaded.")

Prompt templates loaded.


### System Prompt & Per-Task Prompt Templates

Defines the system prompt that instructs the generator model to behave as an expert
PM/agile coach and always return valid JSON with no extra text. Also defines
task-specific prompt-builder functions (e.g. `make_task_design_prompt`) that format
either a real seed or a synthetic scenario into the exact instruction sent to the
Inception API. Keeping prompts in dedicated functions makes it easy to keep the output
JSON schema consistent per task type.


In [16]:
# ============================================================
# INCEPTION API CALL WRAPPER  (rate-limited + retry)
# ============================================================

def call_inception(user_message: str, retries: int = 3) -> str | None:
    """
    Call Inception API with:
    - Sliding-window rate limiting (≤95 req/min)
    - Exponential backoff on failure
    - 429 handling: respects Retry-After header when present
    """
    headers = {
        "Authorization": f"Bearer {INCEPTION_API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": GENERATOR_MODEL,
        "max_tokens": 1200,
        "temperature": 0.8,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
    }

    for attempt in range(retries):
        rate_limiter.acquire()   # blocks if needed to stay under rate limit
        try:
            resp = requests.post(INCEPTION_BASE_URL, headers=headers,
                                 json=payload, timeout=60)

            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"]

            elif resp.status_code == 429:
                # Rate-limited by server — respect Retry-After if present
                retry_after = int(resp.headers.get("Retry-After", 60))
                print(f"  ⚠ 429 Rate Limited. Waiting {retry_after}s (attempt {attempt+1})...")
                time.sleep(retry_after)

            else:
                wait = (attempt + 1) * 10
                print(f"  ⚠ HTTP {resp.status_code} on attempt {attempt+1}. Waiting {wait}s...")
                print(f"     Response: {resp.text[:200]}")
                time.sleep(wait)

        except requests.exceptions.Timeout:
            wait = (attempt + 1) * 15
            print(f"  ⚠ Timeout on attempt {attempt+1}. Waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            wait = (attempt + 1) * 10
            print(f"  ⚠ Error on attempt {attempt+1}: {e}. Waiting {wait}s...")
            time.sleep(wait)

    return None


def parse_json_response(raw: str) -> dict | None:
    """Strip markdown fences if present, then parse JSON."""
    if not raw:
        return None
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip().rstrip("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end   = raw.rfind("}") + 1
        if start != -1 and end > start:
            try:
                return json.loads(raw[start:end])
            except:
                pass
    return None


def format_qwen2_chat(instruction: str, response_dict: dict, task_type: str) -> dict:
    """Format as Qwen2 chat instruction-response pair (ChatML format)."""
    return {
        "id":        hashlib.md5(instruction.encode()).hexdigest()[:12],
        "task_type": task_type,
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": json.dumps(response_dict, indent=2, ensure_ascii=False)},
        ]
    }


print("API wrapper ready.")

API wrapper ready.


### Inception API Call Wrapper

Wraps the raw HTTP call to the Inception API with the rate limiter from the previous
cell, plus retry logic with exponential backoff and explicit handling of HTTP 429
("too many requests") responses, including honoring any `Retry-After` header the API
returns. Centralizing this logic means every generator function below can call
`call_inception()` without worrying about rate limits or transient failures.


In [17]:
# ============================================================
# QUICK TEST — run this BEFORE the full pipeline
# ============================================================

def quick_test():
    print("Running quick API test (1 request)...")
    seed   = REAL_JIRA_SEEDS[0]
    prompt = make_task_design_prompt(seed, is_real=True)
    raw    = call_inception(prompt)
    parsed = parse_json_response(raw)

    if parsed:
        print("✅ Inception API OK! Sample output:")
        print(json.dumps(parsed, indent=2)[:500] + "...")
    else:
        print("❌ API test failed. Check your INCEPTION_API_KEY.")
        print(f"   Raw response: {raw[:300] if raw else 'None'}")
    return parsed is not None

quick_test()

Running quick API test (1 request)...
✅ Inception API OK! Sample output:
{
  "task_id": "TASK-042",
  "title": "Implement Soil Moisture Ingestion Pipeline",
  "description": "Develop a robust, scalable pipeline to collect, validate, and store time-series soil moisture data from 10,000 distributed sensors. The pipeline must handle high-frequency data, ensure data integrity, and integrate with the existing data lake for downstream analytics.",
  "acceptance_criteria": [
    "Pipeline ingests data from all 10k sensors with end-to-end latency of 5 seconds or less",
    "...


True

### Quick Sanity Test

Before committing to the full (potentially 20+ minute) generation run, this cell
fires a single test request end-to-end (prompt construction → API call → JSON
parsing) and reports success/failure. Always run this first to catch a bad/missing
API key early rather than discovering it partway through a long batch job.


In [18]:
# ============================================================
# GENERATOR FUNCTIONS
# ============================================================

def generate_task_design_examples(n: int, real_ratio: float) -> list[dict]:
    examples = []
    n_real   = int(n * real_ratio)
    n_synth  = n - n_real
    print(f"\n[Task Design] {n_real} real-seeded + {n_synth} synthetic = {n} total")

    seeds = (REAL_JIRA_SEEDS * ((n_real // len(REAL_JIRA_SEEDS)) + 1))[:n_real]
    for seed in tqdm(seeds, desc="  Real-seeded"):
        prompt = make_task_design_prompt(seed, is_real=True)
        parsed = parse_json_response(call_inception(prompt))
        if parsed:
            examples.append(format_qwen2_chat(prompt, parsed, "task_design"))

    for _ in tqdm(range(n_synth), desc="  Synthetic"):
        prompt = make_task_design_prompt(random.choice(REAL_JIRA_SEEDS), is_real=False)
        parsed = parse_json_response(call_inception(prompt))
        if parsed:
            examples.append(format_qwen2_chat(prompt, parsed, "task_design"))

    print(f"  ✓ {len(examples)} valid task design examples")
    return examples


def generate_task_allocation_examples(n: int, real_ratio: float) -> list[dict]:
    examples = []
    n_real   = int(n * real_ratio)
    n_synth  = n - n_real
    print(f"\n[Task Allocation] {n_real} real-seeded + {n_synth} synthetic = {n} total")

    seeds = (REAL_JIRA_SEEDS * ((n_real // len(REAL_JIRA_SEEDS)) + 1))[:n_real]
    for seed in tqdm(seeds, desc="  Real-seeded"):
        team   = random.sample(TEAM_ROSTER_POOL, random.randint(4, 7))
        prompt = make_task_allocation_prompt(seed, team, is_real=True)
        parsed = parse_json_response(call_inception(prompt))
        if parsed:
            examples.append(format_qwen2_chat(prompt, parsed, "task_allocation"))

    for _ in tqdm(range(n_synth), desc="  Synthetic"):
        seed   = random.choice(REAL_JIRA_SEEDS)
        team   = random.sample(TEAM_ROSTER_POOL, random.randint(3, 6))
        prompt = make_task_allocation_prompt(seed, team, is_real=False)
        parsed = parse_json_response(call_inception(prompt))
        if parsed:
            examples.append(format_qwen2_chat(prompt, parsed, "task_allocation"))

    print(f"  ✓ {len(examples)} valid task allocation examples")
    return examples


def generate_resource_planning_examples(n: int) -> list[dict]:
    examples = []
    print(f"\n[Resource Planning] Generating {n} examples...")
    for _ in tqdm(range(n), desc="  Generating"):
        prompt = make_resource_planning_prompt(is_real=False)
        parsed = parse_json_response(call_inception(prompt))
        if parsed:
            examples.append(format_qwen2_chat(prompt, parsed, "resource_planning"))
    print(f"  ✓ {len(examples)} valid resource planning examples")
    return examples


def generate_sprint_planning_examples(n: int) -> list[dict]:
    examples = []
    print(f"\n[Sprint Planning] Generating {n} examples...")
    for _ in tqdm(range(n), desc="  Generating"):
        prompt = make_sprint_planning_prompt(is_real=False)
        parsed = parse_json_response(call_inception(prompt))
        if parsed:
            examples.append(format_qwen2_chat(prompt, parsed, "sprint_planning"))
    print(f"  ✓ {len(examples)} valid sprint planning examples")
    return examples


print("Generators ready.")

Generators ready.


### Generator Functions

Defines the functions that produce the actual dataset examples for each task type
(e.g. `generate_task_design_examples`). Each function mixes real-seeded and purely
synthetic prompts according to `real_ratio`, calls the Inception API for each one,
parses the JSON response, and formats it into a Qwen2 chat-style training example.


In [19]:
# ============================================================
# QUALITY VALIDATION
# ============================================================

REQUIRED_KEYS = {
    "task_design":       ["task_id", "title", "description", "acceptance_criteria", "story_points"],
    "task_allocation":   ["assigned_to", "role", "rationale", "estimated_hours"],
    "resource_planning": ["project", "sprints", "total_sprints"],
    "sprint_planning":   ["sprint_goal", "selected_stories", "total_story_points"],
}

def validate_example(example: dict) -> bool:
    task_type = example.get("task_type", "")
    if task_type not in REQUIRED_KEYS:
        return False
    try:
        parsed   = json.loads(example["messages"][2]["content"])
        required = REQUIRED_KEYS[task_type]
        return all(k in parsed for k in required)
    except:
        return False


def validate_dataset(examples: list[dict]) -> tuple[list[dict], dict]:
    valid   = [e for e in examples if validate_example(e)]
    stats   = {
        "total_generated": len(examples),
        "valid":           len(valid),
        "invalid_dropped": len(examples) - len(valid),
        "pass_rate":       f"{len(valid)/max(len(examples),1)*100:.1f}%",
        "by_task_type":    {t: sum(1 for e in valid if e["task_type"] == t) for t in REQUIRED_KEYS},
    }
    return valid, stats


print("Validation helpers ready.")

Validation helpers ready.


### Quality Validation

Defines the required output fields for each task type (`REQUIRED_KEYS`) and a
`validate_example()` function that checks a generated example's JSON payload contains
all the fields expected for its task type. This acts as a data-quality gate — any
example failing validation should be excluded from the final dataset to avoid
polluting fine-tuning/evaluation with malformed labels.


In [20]:
# ============================================================
# TRAIN / VAL / TEST SPLIT + SAVE
# ============================================================

def save_dataset(examples: list[dict], output_dir: Path):
    random.shuffle(examples)
    n       = len(examples)
    n_train = int(n * 0.80)
    n_val   = int(n * 0.10)

    splits = {
        "train": examples[:n_train],
        "val":   examples[n_train:n_train + n_val],
        "test":  examples[n_train + n_val:],
    }

    for split_name, split_data in splits.items():
        path = output_dir / f"pm_{split_name}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for ex in split_data:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        print(f"  Saved {len(split_data):4d} examples → {path}")

    full_path = output_dir / "pm_finetune_dataset.jsonl"
    with open(full_path, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"  Saved {len(examples):4d} examples → {full_path} (full dataset)")

    meta = {
        "dataset_name":     "PM-SFT-Qwen2",
        "version":          "1.0",
        "total_examples":   n,
        "splits":           {k: len(v) for k, v in splits.items()},
        "task_types":       list(REQUIRED_KEYS.keys()),
        "format":           "qwen2-chat (ChatML)",
        "base_model_target":"Qwen/Qwen2-1.7B",
        "generator_model":  GENERATOR_MODEL,
        "generator_api":    "Inception Labs (api.inceptionlabs.ai)",
        "real_jira_ratio":  REAL_RATIO,
        "rate_limit_used":  f"{MAX_REQUESTS_PER_MIN}/min",
        "source_datasets": [
            "Apache Kafka Jira (Apache 2.0)",
            "Spring Framework Jira (Apache 2.0)",
            "Synthetic (Inception API-generated)",
        ],
    }
    with open(output_dir / "dataset_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  Saved metadata → {output_dir}/dataset_metadata.json")
    return splits


def preview_example(examples: list[dict], task_type: str):
    sample = next((e for e in examples if e["task_type"] == task_type), None)
    if not sample:
        print(f"No example for task_type={task_type}")
        return
    print(f"\n{'='*60}")
    print(f"EXAMPLE — {task_type.upper().replace('_', ' ')}")
    print(f"{'='*60}")
    print(f"[USER]\n{sample['messages'][1]['content'][:500]}...")
    print(f"\n[ASSISTANT]\n{sample['messages'][2]['content'][:600]}...")
    print(f"{'='*60}\n")


print("Save helpers ready.")

Save helpers ready.


### Train / Validation / Test Split & Save

Shuffles the full pool of generated (and validated) examples and splits them into an
80/10/10 train/validation/test partition, writing each split to its own `.jsonl` file.
This split is what downstream notebooks consume: the fine-tuning notebook trains on
`pm_train.jsonl`, and the benchmarking/evaluation notebooks score models against
`pm_test.jsonl`.


In [ ]:
# ============================================================
# MAIN PIPELINE
# ============================================================
# At 95 req/min the 2000-example run takes ~21 minutes.
# For a quick smoke test, set EXAMPLES_PER_TASK = 5 first.
# ============================================================

def main():
    print("=" * 60)
    print("PM Dataset Preparation Pipeline — Inception API")
    print(f"Target : {EXAMPLES_PER_TASK * 4} examples | Model: {GENERATOR_MODEL}")
    print(f"Rate   : {MAX_REQUESTS_PER_MIN} req/min  (limit: 100)")
    est_min = (EXAMPLES_PER_TASK * 4) / MAX_REQUESTS_PER_MIN
    print(f"Est.   : ~{est_min:.1f} min (excluding retries)")
    print("=" * 60)

    td  = generate_task_design_examples(EXAMPLES_PER_TASK, REAL_RATIO)
    ta  = generate_task_allocation_examples(EXAMPLES_PER_TASK, REAL_RATIO)
    rp  = generate_resource_planning_examples(EXAMPLES_PER_TASK)
    sp  = generate_sprint_planning_examples(EXAMPLES_PER_TASK)

    all_examples = td + ta + rp + sp

    print("\n[Validation] Running quality checks...")
    valid_examples, stats = validate_dataset(all_examples)
    print("\nValidation Report:")
    print(json.dumps(stats, indent=2))

    for task_type in REQUIRED_KEYS:
        preview_example(valid_examples, task_type)

    print("\n[Saving] Writing dataset files...")
    splits = save_dataset(valid_examples, OUTPUT_DIR)

    print("\n" + "=" * 60)
    print(f"✅ Done! Dataset ready at: {OUTPUT_DIR.resolve()}")
    print(f"   Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")
    print("=" * 60)


# Run pipeline
main()

PM Dataset Preparation Pipeline — Inception API
Target : 2000 examples | Model: mercury-2
Rate   : 95 req/min  (limit: 100)
Est.   : ~21.1 min (excluding retries)

[Task Design] 150 real-seeded + 350 synthetic = 500 total


  Real-seeded:  13%|█▎        | 20/150 [00:21<02:05,  1.04it/s]

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Real-seeded:  27%|██▋       | 40/150 [01:44<02:11,  1.19s/it]

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Real-seeded:  41%|████      | 61/150 [03:09<01:48,  1.22s/it]

... (251 outputs left out)

  Generating:  82%|████████▏ | 411/500 [53:02<03:53,  2.62s/it]

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Generating:  84%|████████▍ | 421/500 [54:22<03:39,  2.78s/it]

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


### Main Pipeline (Full Run)

Orchestrates the end-to-end generation: for each of the four PM task types, calls the
corresponding generator function for `EXAMPLES_PER_TASK` examples, validates them, and
saves the resulting train/val/test splits to disk. Prints a summary of the target
example count, generator model, rate limit, and estimated wall-clock time before
kicking off the run.

**Note:** the next cell is a duplicate of this pipeline with some generation calls
commented out — likely used during development to re-run only a subset of task types
(e.g. after fixing a bug in one generator) without regenerating the entire dataset.


In [ ]:
# ============================================================
# MAIN PIPELINE
# ============================================================
# At 95 req/min the 2000-example run takes ~21 minutes.
# For a quick smoke test, set EXAMPLES_PER_TASK = 5 first.
# ============================================================

def main():
    print("=" * 60)
    print("PM Dataset Preparation Pipeline — Inception API")
    print(f"Target : {EXAMPLES_PER_TASK * 4} examples | Model: {GENERATOR_MODEL}")
    print(f"Rate   : {MAX_REQUESTS_PER_MIN} req/min  (limit: 100)")
    est_min = (EXAMPLES_PER_TASK * 4) / MAX_REQUESTS_PER_MIN
    print(f"Est.   : ~{est_min:.1f} min (excluding retries)")
    print("=" * 60)

    #td  = generate_task_design_examples(EXAMPLES_PER_TASK, REAL_RATIO)
    #ta  = generate_task_allocation_examples(EXAMPLES_PER_TASK, REAL_RATIO)
    rp  = generate_resource_planning_examples(EXAMPLES_PER_TASK)
    sp  = generate_sprint_planning_examples(EXAMPLES_PER_TASK)

    all_examples = rp + sp

    print("\n[Validation] Running quality checks...")
    valid_examples, stats = validate_dataset(all_examples)
    print("\nValidation Report:")
    print(json.dumps(stats, indent=2))

    for task_type in REQUIRED_KEYS:
        preview_example(valid_examples, task_type)

    print("\n[Saving] Writing dataset files...")
    splits = save_dataset(valid_examples, OUTPUT_DIR)

    print("\n" + "=" * 60)
    print(f"✅ Done! Dataset ready at: {OUTPUT_DIR.resolve()}")
    print(f"   Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")
    print("=" * 60)


# Run pipeline
main()

PM Dataset Preparation Pipeline — Inception API
Target : 2000 examples | Model: mercury-2
Rate   : 95 req/min  (limit: 100)
Est.   : ~21.1 min (excluding retries)

[Resource Planning] Generating 500 examples...


  Generating:   2%|▏         | 10/500 [00:28<17:30,  2.14s/it]

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Generating:   4%|▍         | 20/500 [01:49<22:15,  2.78s/it]  

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Generating:   6%|▌         | 30/500 [03:07<19:49,  2.53s/it]  

... (8 outputs left out)

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Generating:  16%|█▌        | 79/500 [09:50<19:49,  2.83s/it]  

  ⚠ 429 Rate Limited. Waiting 60s (attempt 1)...


  Generating:  16%|█▌        | 80/500 [10:52<2:24:05, 20.58s/it]